# Load essentials

In [1]:
from glob import glob
from tqdm import tqdm
import json
import pandas as pd
import numpy as np

img_codes = [path.split('\\')[-1].split('.')[0] for path in glob(r'..\additional_data\transcript_img\*.txt')]

# Load paths of images & vids

In [2]:
vid_paths = glob(r'D:\AIC_LITE\vids\*\*')
img_paths = glob(r'D:\AIC_LITE\kfs\*\*\*\*')
img_path_maps = {code: path for code, path in zip(img_codes, img_paths)}
vid_path_maps = {}
vid_id_maps = {}
for code in img_codes:
    for vid_path in vid_paths:
        vid_id = vid_path.split('\\')[-1].split('.')[0]
        if vid_id not in code: continue
        vid_path_maps[code] = vid_path
        vid_id_maps[code] = vid_id
        break

# Load metadata

In [3]:
met_paths = glob(r'D:\AIC_LITE\metadata\*.json')
map_paths = glob(r'D:\AIC_LITE\mapkfs\*.csv')
mets = {}
for met_path, map_path in tqdm(list(zip(met_paths, map_paths))):
    vid_code = met_path.split('\\')[-1].split('.')[0]
    # Load from metadata
    with open(met_path, 'r', encoding='utf-8') as f:
        met = json.load(f)
    yt_link = met.get('watch_url', '')

    # Load from map
    frame_map = pd.read_csv(map_path)
    for row in list(frame_map.itertuples()):
        frame_id, time_order, fps, frame_order = row.n, row.pts_time, row.fps, row.frame_idx
        code = f'{vid_code}_{frame_id:03d}'
        mets[code] = {
            'frame_id': f'{frame_id:03d}',
            'time_order': time_order,
            'frame_order': frame_order,
            'fps': fps,
            'yt_link': f'{yt_link}&t={time_order}s'
        }

100%|██████████| 31/31 [00:00<00:00, 56.44it/s]


# Load object detection

In [ ]:
obj_paths = glob(r'..\additional_data\obj_det\*.json')
objs = {}
for path, code in tqdm(list(zip(obj_paths, img_codes))):
    det = []
    with open(path, 'r') as f:
        det = json.load(f)
    objs[code] = det

100%|██████████| 8115/8115 [00:00<00:00, 8968.51it/s] 


# Load transcript

In [4]:
tran_paths = glob(r'..\additional_data\transcript_img\*.txt')
trans = {}
for path, code in tqdm(list(zip(tran_paths, img_codes))):
    tran = ''
    with open(path, 'r', encoding='utf-8') as f:
        tran = f.read()
    trans[code] = tran

100%|██████████| 8115/8115 [00:30<00:00, 268.22it/s]


# Load vectors

In [5]:
siglip2_dense_paths = glob(r'..\additional_data\feats\siglip2_dense\*.npy')
bgem3_dense_paths = glob(r'..\additional_data\feats\bgem3_dense\*.npy')
bgem3_sparse_token_id_paths = glob(r'..\additional_data\feats\bgem3_sparse\token_ids\*.npy')
bgem3_sparse_weight_paths = glob(r'..\additional_data\feats\bgem3_sparse\weights\*.npy')

paths_loader = list(zip(
    siglip2_dense_paths,
    bgem3_dense_paths,
    bgem3_sparse_token_id_paths,
    bgem3_sparse_weight_paths
))

vectors = {}
for code, paths in tqdm(list(zip(img_codes, paths_loader))):
    siglip2_dense = np.load(paths[0]).tolist()
    bgem3_dense = np.load(paths[1]).tolist()
    bgem3_sparse = {
        'token_ids': np.load(paths[2]).tolist(),
        'weights': np.load(paths[3]).tolist()
    }

    vectors[code] = {
        'dense_siglip2': siglip2_dense,
        'dense_bgem3': bgem3_dense,
        'sparse_bgem3': bgem3_sparse,
    }

100%|██████████| 8115/8115 [02:43<00:00, 49.75it/s] 


# Update samples accordingly

In [6]:
COLLECTION_NAME = 'test'

for code in tqdm(img_codes):
    point = {
        'vector': vectors[code],
        'payload': {
            'img_path': img_path_maps[code],
            'vid_path': vid_path_maps[code],
            'frame_id': mets[code]['frame_id'],
            'vid_id': vid_id_maps[code],
            'frame_order': mets[code]['frame_order'],
            'time_order': mets[code]['time_order'],
            'fps': mets[code]['fps'],
            'yt_link': mets[code]['yt_link'],
            # 'objs': objs[code],
            'transcript': trans[code],
        }
    }
    with open(fr'..\collections\{COLLECTION_NAME}\{code}.json', 'w') as f:
        json.dump(point, f, indent=4)

100%|██████████| 8115/8115 [00:23<00:00, 350.29it/s]
